In [ ]:
#| default_exp machine_learning.producers.def_and_notat_ner

> **AI-generated — pending human review.** Created by Claude Code on 2026-10-22. Review checklist: `_nbdev_development/AI_REVIEW_CHECKLIST.md` (workspace root). Remove this cell only after review passes.

First producer of the R2 cleanup: dataset assembly for pipeline C (definition/notation token classification) moves from hand-run playground cells (`trouver_personal_playground/front_use_utility_notebooks/ml_data_update/ml_data_update_def_and_notat_token_classification.ipynb`) into a tested library function. Mapping old → new:

| Old playground step | New home |
|---|---|
| local copy of `split_list_consistently` | deleted — `machine_learning.train_split` owns splitting, and a producer does not split |
| `html_data_from_note` per note (skips `_auto/def_and_notat_identified` notes) | unchanged, called here — the existing unverified-auto-marks purity rule is inherited |
| `augment_html_data` mid-extraction (ratio 3–4×) | **not here** (R5) — a span-aware recipe in `ml_augmentation` will do it post-hoc on the saved pure dataset |
| `pickle.dump(list[HTMLData])` | `save_rows` → schema-validated JSONL at `_ml_data/def_and_notat_ner_v1/` |

Rows keep the extraction's tokenizer-independent format — raw text plus `(start, end, label)` char spans — because tokenization happens at training time (`tokenize_html_data`), so one dataset serves any future tokenizer/encoder. The `task` column stamps R9 family membership: producers for note division, link placement, and footnote placement will emit the same row shape, and a multi-task trainer concatenates them.

**v2 (2026-10-22)** adds the per-span `verified` flag. The v1 real-vault timing run (PROGRESS 2026-10-22) showed 78.5% of runtime in the playground's note-level admission loop: it re-extracted every candidate to estimate a verified-mark ratio and traversed See-also notation notes whose only purpose was estimating that same verifiedness. v2 moves the information into the spans themselves — a non-empty `definition`/`notation` attribute is human-authored, an empty one is what auto-marking writes — so membership becomes pure linkage (one extraction pass over all members, no admission re-extraction) and precision filtering becomes `keep_verified_spans`, applied at training time. Measured caveat (600-note sample): ~70% of notation marks carry the ordinary manual form `notation=""`, so `verified=False` does not mean auto-generated for notations; the filter is opt-in and the dataset keeps every span. Membership remains the caller's choice; when regenerating, exclude the 22 notes with on-disk bs4 garbage (`audit/findings_angle_bracket_2026-10-22.md`) — their delimiters are permanently lost (no vault history).

In [ ]:
#| export
from collections.abc import Iterable
from os import PathLike
from pathlib import Path
from typing import Callable

from trouver.machine_learning.ml_data import (PROVENANCE_COLUMN, DatasetSchema, dataset_dir,
                                              save_rows)
from trouver.machine_learning.tokenize.def_and_notat_token_classification import (HTMLData,
                                                                                  def_or_notat_from_html_tag,
                                                                                  html_data_from_note)
from trouver.obsidian.vault import VaultNote

DEF_AND_NOTAT_NER_SCHEMA = DatasetSchema('def_and_notat_ner', 2, ('task', 'Note name', 'text', 'spans'))
"""Canonical schema of the pure C dataset: `spans` is a list of
`{'start', 'end', 'label', 'verified'}` dicts with `label` in
`{'definition', 'notation'}`, char offsets into `text` — tokenizer-independent
by design. v2 added the per-span `verified` flag (`html_data_to_ner_row`)."""

In [ ]:
#| export
def html_data_to_ner_row(html_data: HTMLData,
                         label_from_tag: Callable = def_or_notat_from_html_tag) -> dict:
    """
    Flatten one `HTMLData` datapoint into a JSON-safe `ml_data` row.

    Only the char span, the label name and the mark's verifiedness survive the
    bs4 tag, so the dataset round-trips through JSON instead of pickles. A note
    whose marks carry no relevant label yields an empty `spans` list — a
    legitimate all-negative training row, mirroring the old flow (which
    tokenized such notes with all-`O` labels).

    `verified` is read from the mark's own attribute: non-empty `definition`
    (human-written definition text) or `notation` (human-chosen name) means
    `True`; an empty attribute — what auto-marking writes — means `False`. For
    notation marks an empty attribute is also the ordinary manual form, so
    filtering by the flag is the consumer's choice (`keep_verified_spans`).

    **Parameters**
    - `html_data` (*HTMLData*) — One output of `html_data_from_note`.
    - `label_from_tag` (*Callable*) — Tag → label-name function; `None` means the tag carries no relevant label. Defaults to `def_or_notat_from_html_tag`.
    """
    spans = [{'start': start, 'end': end, 'label': label,
              'verified': bool(str(tag.attrs.get(label, '')).strip())}
             for tag, start, end in html_data['tags']
             if (label := label_from_tag(tag)) is not None]
    return {'task': 'def_and_notat_ner', PROVENANCE_COLUMN: html_data['note_name'],
            'text': html_data['raw_text'], 'spans': spans}


In [ ]:
#| export
def keep_verified_spans(rows: list[dict]) -> list[dict]:
    """
    Copy `rows` keeping only spans with `verified=True`; text and provenance untouched.

    Replaces the old playground's note-level admission loop, which re-extracted
    every candidate to estimate a verified ratio and dropped whole notes: the
    per-span flag carries that information exactly, so producers extract each
    linked member once and precision filtering happens here, at training time.
    Rows left without spans stay as all-negative rows — mirroring the old flow,
    which trained admitted notes' unlabeled text as `O`-labeled tokens.

    **Parameters**
    - `rows` (*list[dict]*) — Rows produced by `produce_def_and_notat_ner_rows`.
    """
    return [{**row, 'spans': [s for s in row['spans'] if s['verified']]} for row in rows]


In [ ]:
#| export
def produce_def_and_notat_ner_rows(notes: Iterable[VaultNote]) -> list[dict]:
    """
    Extract pure definition/notation NER rows from vault `notes`; validated, saved nowhere.

    Notes `html_data_from_note` refuses are skipped: deleted notes and those
    tagged `_auto/def_and_notat_identified` — the existing rule that
    auto-generated, unverified marks never enter training data. Dataset
    membership is the caller's choice (the old playground flow used
    `personal_vault.notes.notes_linked_in_note` of an index note); this reads
    vault notes and touches nothing else — no augmentation (R5), no splitting
    (`train_split`), no writing. Extraction resolves embeds (`![[note#^block]]`)
    through name-based vault lookup, so it is reproducible only with a warm
    the class-level `VaultNote` name->path cache, which only a full walk
    populates (`VaultNote.update_cache(vault)`, or constructing any note by
    name with `update_cache=True`; building instances from `rel_path` does
    NOT register names). On an unpopulated cache lookups miss silently and
    embeds inline as empty text, so warm the cache before producing.

    **Parameters**
    - `notes` (*Iterable[VaultNote]*) — The notes to extract from.
    """
    rows = []
    for note in notes:
        html_data = html_data_from_note(note)
        if html_data is None:
            continue
        rows.append(html_data_to_ner_row(html_data))
    DEF_AND_NOTAT_NER_SCHEMA.validate_rows(rows)
    return rows


def save_def_and_notat_ner_dataset(rows: list[dict], root: PathLike) -> Path:
    """
    Write `rows` (from `produce_def_and_notat_ner_rows`) as the canonical JSONL dataset; return its path.

    **Parameters**
    - `rows` (*list[dict]*) — Produced rows.
    - `root` (*PathLike*) — Usually the vault directory; writes `_ml_data/def_and_notat_ner_v1/def_and_notat_ner.jsonl`.
    """
    path = dataset_dir(root, DEF_AND_NOTAT_NER_SCHEMA) / 'def_and_notat_ner.jsonl'
    save_rows(rows, path)
    return path

In [ ]:
import json
import tempfile
import unittest.mock
from pathlib import Path

import bs4
from fastcore.test import *

from trouver.machine_learning import ml_data as mld
from trouver.machine_learning.producers import def_and_notat_ner as prod

In [ ]:
# Real bs4 tags flatten to JSON-safe spans; unlabeled attrs drop; empty attrs = unverified; empty marks = negative row.
soup = lambda html: bs4.BeautifulSoup(html, 'html.parser')
tag_def = soup('<span definition="1">x</span>').span
tag_not = soup('<span notation="1">y</span>').span
tag_def_auto = soup('<span definition="">x</span>').span
tag_not_manual = soup('<span notation="">y</span>').span
tag_junk = soup('<span class="x">z</span>').span

hd = prod.HTMLData(note_name='n', raw_text='abcde', tags=[(tag_def, 0, 1), (tag_junk, 1, 2),
                                                          (tag_not, 2, 3), (tag_def_auto, 3, 4),
                                                          (tag_not_manual, 4, 5)])
row = html_data_to_ner_row(hd)
test_eq(row['spans'], [{'start': 0, 'end': 1, 'label': 'definition', 'verified': True},
                       {'start': 2, 'end': 3, 'label': 'notation', 'verified': True},
                       {'start': 3, 'end': 4, 'label': 'definition', 'verified': False},
                       {'start': 4, 'end': 5, 'label': 'notation', 'verified': False}])
test_eq(row[mld.PROVENANCE_COLUMN], 'n')
test_eq(row['task'], 'def_and_notat_ner')
json.loads(json.dumps([row]))  # JSON-safe: no bs4 object survives the flattening

neg = html_data_to_ner_row(prod.HTMLData(note_name='n', raw_text='abc', tags=[]))
test_eq(neg['spans'], [])
DEF_AND_NOTAT_NER_SCHEMA.validate_rows([row, neg])

# keep_verified_spans filters spans only: text/provenance stay, originals untouched.
filtered = prod.keep_verified_spans([row, neg])
test_eq([r['spans'] for r in filtered], [row['spans'][:2], []])
test_eq(filtered[0]['text'], row['text'])
test_eq(len(row['spans']), 4)


In [ ]:
# Produce skips refused notes (mocked I/O boundary); save/load round-trips as pure.
good = prod.HTMLData(note_name='a', raw_text=r'Let $\alpha$ be...', tags=[(tag_def, 0, 1)])
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    with unittest.mock.patch.object(prod, 'html_data_from_note', side_effect=[good, None, good]):
        rows = prod.produce_def_and_notat_ner_rows([object(), object(), object()])
    test_eq(len(rows), 2)  # the None (deleted / _auto-tagged) note is skipped

    path = prod.save_def_and_notat_ner_dataset(rows, root)
    assert path.exists() and path.parent.name == 'def_and_notat_ner_v2'
    back = mld.load_rows(path, schema=DEF_AND_NOTAT_NER_SCHEMA, require_pure=True)
    test_eq(back, rows)  # nested spans round-trip through JSONL untouched

## Examples

```python
# Membership as in the old playground flow: an index note links the dataset's notes.
from trouver.personal_vault.notes import notes_linked_in_note
index_note = VaultNote('<name of your ML data index note>', MATH_VAULT_LOCATION)
notes = notes_linked_in_note(index_note, as_dict=False)

rows = produce_def_and_notat_ner_rows(notes)
path = save_def_and_notat_ner_dataset(rows, MATH_VAULT_LOCATION)
# -> <vault>/_ml_data/def_and_notat_ner_v1/def_and_notat_ner.jsonl (pure rows, schema-validated)
```